# 🎯 Das Gesamtbild

## Evaluation + Tuning = Die Software der Zukunft

Wir haben jedes Teil einzeln gesehen. Jetzt fügen wir alles zusammen — und am Ende gibt's ein Quiz! 😎

## Der Moment der Wahrheit

Du hast jetzt alle Bausteine kennengelernt:

- **Notebook 1:** LLMs machen Fehler → Metriken messen Qualität → Prompts können verbessert werden
- **Notebook 2:** Optimizer automatisieren Prompt-Verbesserung
- **Notebook 3:** Deine Daten machen den Unterschied
- **Notebook 4:** Agenten nutzen Tools und können auch optimiert werden

Jetzt setzen wir das Puzzle zusammen und sehen das **Gesamtbild**.


In [ ]:
import sys; sys.path.insert(0, ".")
import dspy
from dspy_tasks.config import configure_dspy
from dspy_tasks.tasks import get_task, list_tasks, list_by_tier, TASK_REGISTRY
from dspy_tasks.actions import run_baseline, run_optimization
from dspy_tasks.visualize import *

MODEL = "github_copilot/gpt-5.1"
configure_dspy(MODEL)


### 🔄 Der komplette Workflow auf einen Blick

So sieht der neue Software-Stack aus:

1. **Du schreibst:** Signatures (was du willst) + Metriken (was gut heisst) + Daten (Beispiele)
2. **Der Optimizer:** Probiert verschiedene Varianten durch — systematisch, nicht zufällig
3. **Das Ergebnis:** Ein optimiertes Modul, das du speichern und in Produktion einsetzen kannst

Das Diagramm zeigt dir diesen Workflow visuell.


In [ ]:
from dspy_tasks.visualize import diagram, diagram_compare

diagram([
    {"label": "Dein Code", "detail": "Signatures + Metriken + Daten", "icon": "📝", "color": "#0078d4"},
    {"label": "Optimizer", "detail": "probiert Varianten", "icon": "⚙️", "color": "#ca5010"},
    {"label": "Evaluator", "detail": "misst Qualität", "icon": "📐", "color": "#ca5010"},
    {"label": "Ergebnis", "detail": "Optimierter Prompt", "icon": "🎯", "color": "#107c10"},
    {"label": "Produktion", "detail": "save & deploy", "icon": "🚀", "color": "#107c10"},
], title="Die komplette Pipeline")

diagram_compare(
    {"title": "Klassische Software", "items": ["Quellcode", "Compiler", "Binary", "Test-Suite", "CI/CD"], "icon": "💻", "color": "#8a8886"},
    {"title": "KI-Software", "items": ["Signatures + Metriken", "Optimizer", "Optimierter Prompt", "Evaluations-Dataset", "Re-Optimierung"], "icon": "🧠", "color": "#0078d4"},
    title="Der neue Software-Stack"
)

## 🏆 Der Cross-Model Showdown

Jetzt lassen wir repräsentative Tasks aus jeder Tier auf verschiedenen Modellen laufen. Das Ergebnis: eine Heatmap die zeigt, wo welches Modell stark oder schwach ist.

### 🤔 Warum ein Showdown?

Bisher hast du immer einzelne Aspekte getestet: Metriken, Optimierung, Domain-Daten, Agenten. Jetzt sehen wir das grosse Bild.

Die Fragen, die wir beantworten:
- Welches Modell ist der **beste Allrounder**?
- Lohnt sich ein teures Modell oder reicht ein günstiges mit Optimierung?
- Bei welchen Tasks macht Optimierung den **grössten Unterschied**?

Die Antworten überraschen oft — manchmal ist das billigste Modell nach Optimierung besser als das teuerste ohne.


### 🏆 Der grosse Vergleich

Wir lassen verschiedene Tasks auf verschiedenen Modellen laufen und sehen das Gesamtbild. Aus jeder Schwierigkeitsstufe (Tier) ist ein repräsentativer Task dabei:

- **Tier 1** (einfach): Sentiment Analysis
- **Tier 2** (mittel): Mathe-Textaufgaben
- **Tier 3** (schwer): Ticket Routing mit Domain-Daten
- **Tier 4** (Agent): Calculator Agent mit Tool-Nutzung

Das zeigt dir: Wo lohnt sich Optimierung am meisten? Und wo stösst auch das beste Modell an seine Grenzen?


In [ ]:
SHOWCASE_TASKS = ["sentiment", "math_word", "ticket_routing", "calculator_agent"]
MODELS_TO_COMPARE = [MODEL]  # Füge weitere Modelle hinzu falls gewünscht

task_names = []
all_scores = []

for task_id in SHOWCASE_TASKS:
    task = get_task(task_id)
    task_names.append(task.name)
    row = []
    for model in MODELS_TO_COMPARE:
        configure_dspy(model)
        print(f"  ⏳ {task.name} auf {model.split('/')[-1]}...", end=" ")
        result = run_baseline(task_id, max_eval=5)
        row.append(result.score)
        print(f"{result.score:.0%}")
    all_scores.append(row)
    print()

model_short = [m.split("/")[-1] for m in MODELS_TO_COMPARE]
fig = heatmap_tasks_models(task_names, model_short, all_scores,
    title="Task × Model Performance")
fig.show()


## 💰 Der ROI von Optimierung

Optimierung kostet ein paar Dutzend LLM-Aufrufe einmalig. Aber die Verbesserung gilt für JEDEN zukünftigen Aufruf. Spiel mit den Reglern und sieh den ROI!

### 📊 Ein konkretes Beispiel

Sagen wir, du hast einen Ticket-Routing-Task:
- **Ohne Optimierung:** 65% Genauigkeit. Bei 1000 Tickets pro Monat werden 350 falsch geroutet.
- **Mit Optimierung:** 88% Genauigkeit. Nur noch 120 falsch geroutet.
- **Kosten der Optimierung:** ~30 API-Calls × $0.003 = weniger als 10 Cent.
- **Ersparnis:** 230 Tickets, die nicht manuell umgeroutet werden müssen.

Die Regler unten lassen dich deine eigenen Zahlen eingeben.


### 💰 Was kostet Optimierung — und was bringt sie?

Spiel mit den Reglern und sieh den Return on Investment live.

Die Rechnung ist einfach:
- **Einmalige Kosten:** Ein paar Dutzend LLM-Aufrufe für die Optimierung
- **Dauerhafter Nutzen:** Jede zukünftige Anfrage profitiert vom besseren Prompt

Stell dir vor: 30 API-Calls für die Optimierung vs. tausende bessere Antworten danach. Ab wann lohnt es sich? Finde es mit den Reglern heraus.


In [ ]:
from dspy_tasks.visualize import cost_roi_chart

# ROI Parameter (ändere die Werte und führe die Zelle erneut aus)
OPTIMIZATION_CALLS = 30     # Wie viele LLM-Aufrufe für die Optimierung
COST_PER_CALL = 0.003       # Kosten pro Aufruf in $
BASELINE_ACCURACY = 0.65    # Accuracy ohne Optimierung
OPTIMIZED_ACCURACY = 0.88   # Accuracy nach Optimierung
TOTAL_QUERIES = 10000       # Wie viele Anfragen in Produktion

fig = cost_roi_chart(
    OPTIMIZATION_CALLS, COST_PER_CALL,
    BASELINE_ACCURACY, OPTIMIZED_ACCURACY,
    TOTAL_QUERIES, COST_PER_CALL,
)
fig.show()

opt_cost = OPTIMIZATION_CALLS * COST_PER_CALL
extra_correct = int(TOTAL_QUERIES * (OPTIMIZED_ACCURACY - BASELINE_ACCURACY))
print(f"💰 Optimierungskosten: ${opt_cost:.2f} (einmalig)")
print(f"📈 Zusätzlich korrekte Antworten: {extra_correct:,} bei {TOTAL_QUERIES:,} Anfragen")


## 🔄 Der neue Software-Stack

| Klassische Software | KI-Software |
|---|---|
| Quellcode | Signatures + Metriken + Daten |
| Compiler | Optimizer (z.B. BootstrapFewShot / MIPROv2) |
| Binary | Optimierter Prompt + Few-Shot Beispiele |
| Test-Suite | Evaluations-Dataset |
| CI/CD | Re-Optimierungs-Pipeline |
| Refactoring | Re-Kompilieren mit neuen Daten/Modell |

### 🔄 Was heisst das konkret?

| Klassische Software | KI-Software |
|---|---|
| Du schreibst Code | Du schreibst Signatures + Metriken |
| Compiler optimiert | Optimizer optimiert |
| Du deployest eine Binary | Du deployest ein optimiertes Modul |
| Bug? → Code ändern | Schlechte Ergebnisse? → Metrik anpassen, Daten erweitern |

Der Paradigmenwechsel: **Du programmierst nicht mehr das Verhalten direkt — du spezifizierst was du willst und lässt den Optimizer den besten Weg finden.**


## 💾 Produktions-Patterns

In Produktion speicherst du das optimierte Modul mit `module.save('pfad.json')` und lädst es mit `module.load('pfad.json')`. Die Optimierungskosten fallen einmalig an — der Nutzen gilt für jede zukünftige Anfrage.

### 💡 Warum ist Speichern & Laden so wichtig?

Optimierung ist *einmalig teuer* (ein paar Dutzend API-Calls), aber der Nutzen ist *dauerhaft*. Du optimierst einmal, speicherst das Ergebnis als JSON-Datei, und lädst es in Produktion.

```python
# So einfach geht's:
optimized_module.save('mein_modul.json')   # einmalig
loaded = MyModule(); loaded.load('mein_modul.json')  # in Produktion
```

Wenn sich deine Daten ändern (neue Tickets, neue Kategorien), optimierst du einfach nochmal. Der Prozess ist reproduzierbar — wie ein neuer Build in der CI/CD-Pipeline.


In [ ]:
display_insight("Speichern & Laden optimierter Module",
    "In Produktion speicherst du das optimierte Modul mit module.save('pfad.json') "
    "und lädst es mit module.load('pfad.json'). Die Optimierungskosten fallen einmalig an; "
    "der Nutzen gilt für jede zukünftige Anfrage.",
    icon="💾")

## 🎯 Die Pointe

### 🎯 Lies das nochmal langsam

Die nächste Zelle zeigt **die** zentrale Erkenntnis dieser ganzen Workshop-Reihe. Nimm dir einen Moment und lies sie in Ruhe.

Das ist der Satz, der alles zusammenfasst — die eine Idee, die du aus diesem Workshop mitnehmen solltest, selbst wenn du alles andere vergisst.


In [ ]:
from IPython.display import display, Markdown

display(Markdown("""
---
## 🎯 Die Pointe

> **Evaluation** ist die Spezifikation.
>
> **Optimierung** ist der Compiler.
>
> **Daten** sind der Quellcode.

*Willkommen bei Software 3.0.*

---
"""))

## 🧠 Quiz: Was hast du gelernt?

Jetzt testen wir, was hängen geblieben ist! 7 Fragen zu den wichtigsten Konzepten aus allen Notebooks. Viel Erfolg!

### 📋 Was wird abgefragt?

Die Fragen decken alle fünf Notebooks ab:
- Was ist eine Signature? (Notebook 1)
- Wie funktionieren Metriken? (Notebook 1 + 2)
- Was macht der Optimizer? (Notebook 2)
- Warum sind Domain-Daten wichtig? (Notebook 3)
- Wie arbeiten Agenten? (Notebook 4)
- Wie sieht der Produktions-Workflow aus? (Notebook 5)

Nimm dir Zeit — das Quiz ist zum Lernen da, nicht zum Stressen.


### 🧠 Jetzt testen wir, was hängen geblieben ist!

7 Fragen zu den wichtigsten Konzepten aus allen Notebooks. Keine Sorge, es gibt keine Noten 😄

Versuch die Fragen zuerst ohne nachzuschauen — du wirst überrascht sein, wie viel du schon weisst!

**Tipp:** Wenn du bei einer Frage unsicher bist, geh zurück zum entsprechenden Notebook und schau nochmal nach. Das ist kein Mogeln — das ist Lernen.


In [ ]:
from dspy_tasks.visualize import quiz

quiz([
    {
        "question": "Was ist eine dspy.Signature?",
        "options": [
            "Ein Prompt den du von Hand schreibst",
            "Eine reine Daten-Deklaration die beschreibt was rein- und rausgeht",
            "Eine Python-Funktion die das LLM aufruft",
            "Ein API-Schlüssel für das Modell",
        ],
        "answer": 1,
        "explanation": "Signatures sind DATA — reine Deklarationen ohne Verhalten (Notebook 01).",
    },
    {
        "question": "Warum sind Metriken 'Calculations' im Sinne von Grokking Simplicity?",
        "options": [
            "Weil sie das LLM aufrufen",
            "Weil sie Seiteneffekte haben",
            "Weil sie reine Funktionen sind: gleicher Input = gleicher Output, kein I/O",
            "Weil sie nur in Jupyter funktionieren",
        ],
        "answer": 2,
        "explanation": "Metriken sind pure Functions — testbar ohne API-Key, deterministisch (Notebook 01).",
    },
    {
        "question": "Was ist der Unterschied zwischen dspy.Predict und dspy.ChainOfThought?",
        "options": [
            "Predict ist schneller und besser",
            "ChainOfThought fügt automatisch einen Reasoning-Schritt hinzu",
            "Es gibt keinen Unterschied",
            "Predict nutzt Tools, ChainOfThought nicht",
        ],
        "answer": 1,
        "explanation": "ChainOfThought ist ein 'tieferes Modul' — gleiches Interface, aber mit Denkschritt (Notebook 02).",
    },
    {
        "question": "Was macht der DSPy Optimizer?",
        "options": [
            "Er schreibt Python-Code",
            "Er sucht automatisch nach besseren Prompts und Few-Shot-Beispielen",
            "Er trainiert das LLM-Modell neu",
            "Er übersetzt Prompts in andere Sprachen",
        ],
        "answer": 1,
        "explanation": "Der Optimizer ist wie ein Compiler: Signature + Metrik + Daten → optimierter Prompt (Notebook 04).",
    },
    {
        "question": "Warum sind 'deine Daten dein Burggraben'?",
        "options": [
            "Weil Daten teuer sind",
            "Weil ein generisches Modell + deine Domain-Daten + Tuning etwas erzeugt, das kein Konkurrent kopieren kann",
            "Weil man ohne Daten kein LLM nutzen kann",
            "Weil Daten in der Cloud gespeichert werden",
        ],
        "answer": 1,
        "explanation": "Domain-spezifisches Tuning mit eigenen Daten schafft einen Wettbewerbsvorteil (Notebook 05).",
    },
    {
        "question": "Was optimiert DSPy bei Agenten (ReAct)?",
        "options": [
            "Nur die Worte im Prompt",
            "Die Geschwindigkeit der API-Aufrufe",
            "Nicht nur den Prompt, sondern auch WIE der Agent Tools einsetzt",
            "Die Anzahl der verfügbaren Tools",
        ],
        "answer": 2,
        "explanation": "DSPy optimiert die Entscheidungsstrategie des Agenten — wann und wie er Tools nutzt (Notebook 06).",
    },
    {
        "question": "Was ist 'Software 3.0'?",
        "options": [
            "Die neueste Version von Python",
            "Evaluation = Spezifikation, Optimierung = Compiler, Daten = Quellcode",
            "Ein neues Betriebssystem",
            "Software die nur mit GPT-4o funktioniert",
        ],
        "answer": 1,
        "explanation": "Software 3.0: Du schreibst Metriken (Specs), der Optimizer kompiliert Prompts, deine Daten sind der Source Code.",
    },
])

## 🎓 Und jetzt?

Du hast die Grundlagen verstanden. Hier sind deine nächsten Schritte:

1. **Eigene Daten** — Bring deine eigenen Datasets mit und optimiere darauf
2. **Neue Tasks** — Erweitere die `dspy_tasks/tasks/` mit eigenen Aufgaben
3. **Andere Modelle** — Probier verschiedene Modelle aus und vergleiche
4. **Produktion** — Speichere optimierte Module und deploye sie

### 📚 Optionale Appendix-Notebooks

- **Appendix A: Grokking Simplicity** — Wie man LLM-Code in Data, Calculations und Actions aufteilt
- **Appendix B: Deep Modules** — Warum `Predict`, `ChainOfThought` und `ReAct` das gleiche Interface haben

*Evaluation ist die Spezifikation. Optimierung ist der Compiler. Daten sind der Quellcode.* 🚀

### 🎉 Gratulation!

Du hast alle 5 Notebooks durchgearbeitet. Du verstehst jetzt:

- ✅ Warum LLMs Fehler machen und wie man sie mit Metriken misst
- ✅ Wie automatische Prompt-Optimierung funktioniert und warum sie besser ist als manuelles Tuning
- ✅ Warum deine Domain-Daten dein Wettbewerbsvorteil sind — dein Burggraben
- ✅ Wie Agenten Tools nutzen und optimiert werden können
- ✅ Wie das alles in Produktion zusammenspielt — vom Prototyp zum Deployment

**Du bist jetzt kein Prompt Engineer mehr — du bist ein KI-Software-Ingenieur.** 🚀

Teile dein Wissen mit deinem Team. Die Notebooks stehen dir jederzeit als Referenz zur Verfügung.
